# Encoding Categorical Variables: Label Encoding vs One-Hot Encoding
**Summer of Science 2026 — CS03: Artificial Intelligence and Machine Learning**  
**Mohit Khyalia | IIT Bombay**

---

## Introduction

This is a topic I used without really questioning it in Week 1 — every mini-project that had categorical columns just used `LabelEncoder`, because that's what the course material did. This notebook is me actually slowing down and working out *why* that might not always be the right call, and what one-hot encoding does differently.

Initially I assumed Label Encoding was just "the standard way to turn text into numbers" and didn't think there was much more to it. That assumption fell apart pretty quickly once I tried feeding the encoded numbers into something that cares about magnitude (like a distance calculation) and realised the encoder doesn't know or care whether the categories actually have any order.

## Learning Objectives
- Use `LabelEncoder` and understand exactly what numbers it assigns and why
- Use one-hot encoding via `pd.get_dummies()`
- Understand the practical difference between ordinal and nominal categorical data
- See a concrete example of where Label Encoding can mislead a model
- Decide which encoding to use for different kinds of categorical columns

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Imports done')

## Step 1: Label Encoding — The Mechanics

In [ ]:
df = pd.DataFrame({
    'education':     ['Undergraduate', 'Graduate', 'Postgraduate', 'Graduate', 'Undergraduate'],
    'property_area': ['Urban', 'Semiurban', 'Rural', 'Urban', 'Rural'],
    'loan_status':   ['Y', 'N', 'Y', 'Y', 'N']
})
print(df)

In [ ]:
le_education = LabelEncoder()
df['education_encoded'] = le_education.fit_transform(df['education'])

print(df[['education', 'education_encoded']])
print('\nClasses (in the order LabelEncoder assigned numbers):')
print(le_education.classes_)

**Expected output:**
```
       education  education_encoded
0  Undergraduate                  2
1       Graduate                  0
2   Postgraduate                  1
3       Graduate                  0
4  Undergraduate                  2

Classes (in the order LabelEncoder assigned numbers):
['Graduate' 'Postgraduate' 'Undergraduate']
```

One thing I noticed here — and this genuinely confused me the first time — is that `LabelEncoder` assigns numbers **alphabetically**, not based on any logical ordering of the categories. Graduate got 0, Postgraduate got 1, Undergraduate got 2. If I'd assumed the encoder understood that Postgraduate > Graduate > Undergraduate in terms of education level, I'd have been wrong — it's just sorting the unique strings alphabetically and numbering them in that order. It has no concept of what the words actually mean.

## Step 2: Where This Becomes a Real Problem

In [ ]:
le_area = LabelEncoder()
df['property_area_encoded'] = le_area.fit_transform(df['property_area'])

print(df[['property_area', 'property_area_encoded']])
print('\nClasses:', le_area.classes_)

**Expected output:**
```
  property_area  property_area_encoded
0         Urban                       2
1     Semiurban                       1
2         Rural                       0
3         Urban                       2
4         Rural                       0

Classes: ['Rural' 'Semiurban' 'Urban']
```

In [ ]:
# Demonstrate the actual problem: a model using raw numeric distance
# would think Rural (0) and Urban (2) are "further apart" than
# Rural (0) and Semiurban (1) - even though there's no real notion
# of distance between these categories at all

print('Numeric "distance" between categories, as a model would see it:')
print(f'  |Rural - Semiurban| = |0 - 1| = {abs(0-1)}')
print(f'  |Rural - Urban|     = |0 - 2| = {abs(0-2)}')
print(f'  |Semiurban - Urban| = |1 - 2| = {abs(1-2)}')
print()
print('This implies Rural and Urban are "twice as different" as Rural and Semiurban.')
print('But Rural, Semiurban, and Urban have no real ordering - this is a made-up relationship')
print('that only exists because of how LabelEncoder happened to number them alphabetically.')

**Observation:** This became clearer when I actually wrote it out like this rather than just reading the warning in the documentation. A linear model fitting a single weight to the `property_area_encoded` column is implicitly assuming a straight-line relationship between the category number and the outcome — which makes sense for something like education level (where Postgraduate genuinely "is more" than Undergraduate in some sense) but makes no sense at all for Rural/Semiurban/Urban, where the categories are just different labels with no inherent order.

## Step 3: One-Hot Encoding as the Alternative

In [ ]:
df_onehot = pd.get_dummies(df[['property_area']], columns=['property_area'])
print(df_onehot)

**Expected output:**
```
   property_area_Rural  property_area_Semiurban  property_area_Urban
0                 False                    False                 True
1                 False                     True                False
2                  True                    False                False
3                 False                    False                 True
4                  True                    False                False
```

Instead of one column with a number that implies an ordering, one-hot encoding creates a separate True/False column per category. Now there's no implied distance between Rural and Urban — they're just two independent flags, and a model treats them as completely separate dimensions rather than points on a number line.

In [ ]:
# drop_first=True avoids redundancy - if you know it's not Rural and not Semiurban,
# it must be Urban, so you don't need a third column to represent that
df_onehot_dropfirst = pd.get_dummies(df[['property_area']], columns=['property_area'], drop_first=True)
print('With drop_first=True:')
print(df_onehot_dropfirst)

print('\nColumns saved:', df_onehot.shape[1] - df_onehot_dropfirst.shape[1])

**Observation:** I hadn't thought about `drop_first` until I noticed the column count growing fast on a column with many categories. For a feature with only 3 categories it barely matters, but for something like Cabin deck in the Spaceship Titanic dataset (which the Project Brief says has multiple decks), one-hot encoding without dropping a column could add a lot of extra columns. Worth remembering for later.

## Step 4: Visual Comparison of the Two Encodings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Label encoding as a single column
axes[0].bar(range(len(df)), df['property_area_encoded'], color='#1F3864', alpha=0.85)
axes[0].set_xticks(range(len(df)))
axes[0].set_xticklabels(df['property_area'], rotation=20)
axes[0].set_ylabel('Encoded value')
axes[0].set_title('Label Encoding\n(implies an ordering: 0 < 1 < 2)')

# One-hot encoding as a heatmap-style grid
onehot_numeric = df_onehot.astype(int)
im = axes[1].imshow(onehot_numeric.T, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[1].set_yticks(range(len(onehot_numeric.columns)))
axes[1].set_yticklabels(onehot_numeric.columns, fontsize=8)
axes[1].set_xticks(range(len(df)))
axes[1].set_xlabel('Row index')
axes[1].set_title('One-Hot Encoding\n(no ordering — independent flags)')

plt.tight_layout()
plt.savefig('encoding_comparison.png', dpi=150)
plt.show()

## Step 5: A Case Where Label Encoding Actually Makes Sense

In [ ]:
# Not all categorical columns are "unordered" - education level genuinely has an order
education_order = ['Undergraduate', 'Graduate', 'Postgraduate']

df['education_ordinal'] = df['education'].map({cat: i for i, cat in enumerate(education_order)})
print(df[['education', 'education_ordinal']])
print()
print('This is technically still "label encoding" in effect, but with the ordering')
print('chosen deliberately (Undergraduate=0, Graduate=1, Postgraduate=2) rather than')
print('alphabetically. The difference matters: this version actually reflects something real.')

**Observation:** What surprised me here is that the *tool* (LabelEncoder) isn't really the problem — it's using it blindly on every categorical column without checking if the category actually has a meaningful order. For education level, encoding it as an ordered scale genuinely makes sense, since Postgraduate really is "more" than Undergraduate in a way a model might reasonably use. For Property_Area or HomePlanet, there's no such ordering, so one-hot is the safer choice. This feels like the actual takeaway from this notebook — it's not "Label Encoding bad, One-Hot good", it's "check whether the category has a real order before picking."

---

## Summary

| Encoding | What it does | Use when |
|---|---|---|
| `LabelEncoder` (alphabetical) | Assigns 0, 1, 2... in alphabetical order | Quick prototyping, or models that don't care about magnitude (tree-based models) |
| `LabelEncoder` (manual mapping) | Assigns numbers in a deliberately chosen order | Genuinely ordinal categories (education level, size: S/M/L) |
| `pd.get_dummies()` (one-hot) | Creates a separate binary column per category | Unordered (nominal) categories like area, planet, colour |

## Personal Takeaway

Going into this notebook I thought of Label Encoding and One-Hot Encoding as basically interchangeable preprocessing steps. What changed my mind was working out the actual "distance" implications in Step 2 — printing out that Rural and Urban end up "twice as far apart" as Rural and Semiurban, purely because of alphabetical ordering, made the risk concrete rather than theoretical. Going forward, I'm planning to actually check each categorical column and ask whether the categories have a real order before defaulting to Label Encoding, instead of using it everywhere out of habit the way I did in Week 1's mini-projects. This is especially relevant for HomePlanet and Destination in the Spaceship Titanic dataset, since neither has any natural ordering.

---
*Notebook — Mohit Khyalia, Summer of Science 2026, IIT Bombay*